<a href="https://colab.research.google.com/github/saicharan86/AI-Agents/blob/main/Project_1_HBR_Apple_RAG_Assignment_SaiAmujala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

## Business Context

As organizations grow and scale, they are often inundated with large volumes of data, reports, and documents that contain critical information for decision-making. In real-world business settings, such as venture capital firms like Andreesen Horowitz, business analysts are required to sift through large datasets, research papers, or reports to extract relevant information that impacts strategic decisions.

For instance, consider that you've just joined Andreesen Horowitz, a renowned venture capital firm, and you are tasked with analyzing a dense report like the Harvard Business Review's **"How Apple is Organized for Innovation."** Going through the report manually can be extremely time-consuming as the size and complexity of these report increases. However, by using **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models, you can significantly streamline this process.

Imagine having the capability to directly ask questions like, “How does Apple structure its teams for innovation?” and get immediate, relevant answers drawn from the report. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

## Objective

The goal is to develop a RAG application that helps business analysts efficiently extract key insights from extensive reports, such as “How Apple is Organized for Innovation.”

Specifically, the system aims to:

- Answer user queries by retrieving relevant content directly from lengthy documents.

- Support natural-language interaction without requiring a full manual read-through.

- Act as an intelligent assistant that streamlines the report analysis process.

Through this solution, analysts can save time, improve productivity, and make faster, more informed strategic decisions

## Data Description

“How Apple is Organized for Innovation” is a detailed Harvard Business Review article that examines Apple’s unique approach to structuring teams, driving innovation, and maintaining a culture of excellence. The article is provided as a PDF consisting of 11 pages, offering in-depth insights into Apple’s organizational design, leadership principles, and decision-making processes.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30 \
              langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## Imports and environment setup

In [1]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

#### Loading the model

In [2]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                      # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()

In [3]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article?

In [4]:
question_1 = "Who are the authors of this article and who published this article?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"I'm sorry, but I don't have access to external databases or specific articles, so I can't provide information about the authors or the publisher of an article without more context. If you provide me with the title of the article or other details, I might be able to assist you further!"

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [5]:
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
base_prompt_response_2 = response(question_2)
base_prompt_response_2

'- **Integrity**: Leaders with integrity are honest and ethical, earning the trust and respect of their team. This characteristic fosters a positive work environment and encourages open communication.\n\n- **Empathy**: Empathetic leaders understand and relate to the feelings and perspectives of their team members. This ability to connect on a personal level enhances collaboration and boosts morale.\n\n- **Vision**: A strong leader possesses a clear vision for the future, inspiring and motivating their team toward common goals. This characteristic helps in setting a direction and aligning efforts for success.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [6]:
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"I don’t have access to specific articles or their content, but I can provide some general examples of how Apple's leadership approach has fostered successful innovations based on widely known information.\n\n1. **Visionary Leadership of Steve Jobs**: Steve Jobs' leadership style was pivotal in driving Apple's innovation. His focus on design and user experience led to the development of groundbreaking products like the iPod, iPhone, and iPad. Jobs emphasized simplicity and elegance in product design, which helped set Apple apart from competitors.\n\n2. **Cross-Disciplinary Collaboration**: Apple encourages collaboration across different departments, fostering an environment where ideas can be shared freely. This approach was critical in the development of the Apple Watch, where the integration of hardware, software, and health technologies required input from various teams.\n\n3. **Focus on Ecosystem**: Apple's strategy of creating a cohesive ecosystem around its products has led to in

## Observations:

** Question 1**:

The model fails completely to answer question 1 because it requires document-specific knowledge, which is not in its immediate context. It defaults to a safe fallback response, indicating lack of access rather than attempting an answer.


** Question 2**:
The model gives an answer, but it’s generic and not based on the actual article. It uses general knowledge from training instead of information from the document.


** Question 3**:

The model recognizes its limitation but still gives a general answer. However, the response isn’t based on the article, even though the question specifically requires it.



## Question Answering using LLM with Prompt Engineering

Defining a system prompt that aligns with the business problem

In [7]:
system_prompt = """
You are an expert Business Analysis Assistant. Your primary role is to help business analysts extract, interpret, and synthesize insights from business reports and documents based on LLM training data.

Follow these guidelines when answering:
- When Answering, prioritize factual correctness, align with widely accepted standards, and ensure clarity for  users.
- If a query requires specific reference materials, Reference the source contrnt explicitly rather than speculating.
- If the query is vague, infer the most likely business intent and answer accordingly.
- Summarize key points
- Provide structured, business-friendly explanations
- Keep the tone and style professional, analytical and concise

"""

Defining the function to Generate a Response From the LLM

In [8]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article?

In [9]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

'To assist you accurately, I would need more specific information about the article you are referring to, such as its title or subject matter. Please provide additional details so I can help you identify the authors and the publisher.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [10]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

'Here are three key leadership characteristics:\n\n- **Vision**: A strong leader possesses the ability to create a clear and compelling vision for the future, guiding teams towards common goals and inspiring them to achieve ambitious outcomes.\n\n- **Integrity**: Integrity involves being honest and ethical in decision-making, which builds trust and credibility among team members and stakeholders, fostering a positive organizational culture.\n\n- **Empathy**: Empathetic leaders understand and value the perspectives and emotions of their team, enabling them to build strong relationships and enhance team collaboration and morale.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [11]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"To provide a precise analysis of Apple's approach to leadership and its impact on successful innovations, I would need specific details or excerpts from the article you are referencing. However, I can summarize general examples of how Apple's leadership has historically fostered innovation based on widely recognized practices and outcomes.\n\n### Key Leadership Approaches at Apple:\n\n1. **Visionary Leadership**:\n   - **Steve Jobs' Vision**: Steve Jobs emphasized a clear vision that centered around user experience and design simplicity. His insistence on creating products that blend technology with art led to iconic innovations such as the iPhone and iPad.\n\n2. **Cross-Functional Collaboration**:\n   - **Team Integration**: Apple's leadership encourages collaboration across various departments—design, engineering, and marketing. This integration has led to innovations like the seamless ecosystem of devices (iOS, macOS, watchOS) that work harmoniously together.\n\n3. **Focus on Quali

## Observations

** Question 1**:

The model fails completely to answer because it requires document-specific knowledge, which is not in its immediate context.
It defaults to a safe fallback response, indicating lack of access rather than attempting an answer.
This highlights a key limitation: LLMs without RAG cannot reliably retrieve factual metadata from specific documents.
This is a high-confidence retrieval-dependent question, making it ideal for demonstrating RAG value.

** Question 2**:

The model does provide an answer, but it is generic and not grounded in the actual article.
It relies on pre-trained general leadership knowledge, not the document.
There is a hallucination risk—the characteristics may not match the article at all.
This type of question exposes how LLMs can sound correct but be contextually wrong without retrieval.

** Question 3**:

The model partially acknowledges the limitation but still proceeds to generate a generic answer.
The response is not grounded in the article, despite the question explicitly requiring it.
This shows inconsistent behavior:
Q1 → strict refusal
Q3 → fallback + hallucinated generalization
This is a classic example of “confident but ungrounded reasoning”, which RAG is designed to fix.

## Data Preparation for RAG

### Loading the Data

In [12]:
# Set the path to the PDF file
manual_pdf_path = "/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf"      # Path to the HBR Apple PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
document = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

In [13]:
print(f"Loaded {len(document)} pages from the PDF.")
print("First page preview:\n")
print(document[0].page_content[:1200])

Loaded 11 pages from the PDF.
First page preview:

REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.


### Data Chunking

In [14]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=800,                                                             # Each chunk will have up to 800 tokens
    chunk_overlap=100                                                           # 100 tokens will overlap between consecutive chunks (for context continuity)
)

document_chunks = text_splitter.split_documents(document)
print(f"Created {len(document_chunks)} chunks.")
print("Example chunk:\n")
print(document_chunks[0].page_content[:1200])

Created 17 chunks.
Example chunk:

REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.


### Embedding

In [17]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


In [18]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)
print(len(embedding_1))
print(len(embedding_2))

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

1536
1536


([0.0025246840760870756,
  -0.005738068657514522,
  -0.0031388859895014093,
  -0.011899953368824125,
  0.017455913995005092,
  0.030461761753468375,
  -0.010211312720124025,
  0.018065149863373844,
  -0.013694548729464963,
  -0.021495408516744415,
  0.0029650553207922425,
  0.009257726831915744,
  -0.007906813567897566,
  -0.006651921528745025,
  -0.03361389208196238,
  0.01983987824986448,
  0.024091280332840033,
  -0.012893272274010182,
  -0.000846803870741817,
  -0.0019022045672176995,
  -0.03361389208196238,
  -0.0015065328501972029,
  -0.004450066236541811,
  0.009171639094125206,
  0.008939864558738778,
  0.0004747233706173879,
  0.015972559203706237,
  -0.02084644056272051,
  0.029243291879376118,
  -0.0057976681829811855,
  0.01960148201064307,
  0.017204274348113707,
  -0.011701290146580487,
  0.001273103005267355,
  -0.009847095726134295,
  -0.025746810599719967,
  -0.007986279601853118,
  -0.02309796142765398,
  0.024912423762445016,
  -0.0384083074070211,
  0.04071280562792

### Vector Database

Setup Vector Store Directory

In [19]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'research_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

Create Vector Store From Documents

In [20]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

Load Vector Store


In [21]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipykernel_2189/4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Exploring Vector store

In [22]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7e7ed6e36db0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7e7ed6e37ad0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX1/ivhBg0hPOmb1g+QcoqYgIi5a9oyl54fDUxzNQwPHuoKJiPrLR9xwv', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [25]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
   # "Which specific management action did Steve jobs take in his first year back to unify Apple's departments?",
   "who are the authors of the article?",
    k=3
)

[Document(metadata={'modDate': 'D:20201201183749Z', 'creationdate': '2020-10-05T14:18:42-04:00', 'title': '', 'producer': 'Adobe PDF Library 15.0 (via http://bfo.com/products/pdf?version=2.23.5-r33279)', 'format': 'PDF 1.6', 'creationDate': "D:20201005141842-04'00'", 'page': 2, 'total_pages': 11, 'keywords': '', 'moddate': '2020-12-01T18:37:49+00:00', 'subject': '', 'author': '', 'trapped': '', 'source': '/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'encryption': 'Standard V2 R3 128-bit RC4', 'file_path': '/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'creator': 'Adobe InDesign 14.0 (Macintosh)'}, page_content='PHOTOGRAPHER\u2002MIKAEL JANSSON\nHow Apple Is \nOrganized \nfor Innovation\nIt’s about experts \nleading experts.\nORGANIZATIONAL \nCULTURE\nJoel M. \nPodolny\nDean, Apple \nUniversity\nMorten T. \nHansen\nFaculty, Apple \nUniversity\nAUTHORS\nFOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG\nHarvard Business Review\nNovember–Decem

### Retriever

Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [26]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template

In [30]:
# Define the system prompt for the model
qna_system_message = """
You are a Business Analysis Assistant.
Use only the context below to answer the user question.
If the answer is not in the context, say: "I could not find that in the report."

Context:
{context}

Question:
{question}

Answer in clear, simple business English.
"""

In [31]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from the articl that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [32]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article?

In [33]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

'The authors of the article are Joel M. Podolny and Morten T. Hansen. The article was published in the Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [35]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

'- **Deep Expertise**: Apple leaders are experts in their specific fields, allowing them to make informed decisions and provide better guidance to their teams.\n\n- **Immersion in the Details**: Leaders are expected to understand the intricacies of their organization at a deep level, ensuring effective decision-making and addressing issues swiftly.\n\n- **Willingness to Collaboratively Debate**: Managers engage in open discussions with their peers across functions, fostering a collaborative environment that enhances innovation and problem-solving.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [36]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'One specific example from the article is Roger Rosner, who leads Apple’s software application business. His deep expertise in engineering, gained from his background and experience at Apple, allows him to effectively guide teams of experts, fostering an environment where innovation can thrive.\n\nAnother example is the team of over 600 experts on camera hardware technology led by Graham Townsend. This specialized group collaborates closely, ensuring that their collective expertise is utilized effectively. By keeping these experts together rather than scattering them across different product lines, Apple enhances their ability to solve problems and generate innovations.\n\nAdditionally, Apple’s leaders are noted for their deep immersion in the details of their products. This includes their insistence on specific design elements, such as the continuous curves in product design that improve aesthetics and functionality. This attention to detail at the leadership level empowers teams to i

## Actionable Insights and Business Recommendations

-
-
-


<font size=6 color='#4682B4'>Power Ahead</font>
___